## Load libraries

In [2]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.4 MB/s eta 0:00:00


In [10]:
pip install -q sentence-transformers faiss-cpu transformers torch

In [81]:
!pip install -q -U transformers accelerate

## Import Libraries

In [86]:
import os
import requests
import fitz
import faiss
import numpy as np
import pickle
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer , AutoModelForCausalLM

## Create a Knowledge Base

In [4]:
os.makedirs("ml_documents", exist_ok=True)

documents = {
    "attention_is_all_you_need.pdf":
        "https://arxiv.org/pdf/1706.03762",
}

for filename, url in documents.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(os.path.join("ml_documents", filename), "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {filename}")

Downloaded: attention_is_all_you_need.pdf


In [5]:
url = "https://d2l.ai/d2l-en.pdf"
filename = "ml_documents/dive_into_deep_learning.pdf"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(filename, "wb") as f:
    f.write(response.content)

print("Downloaded:", filename)

Downloaded: ml_documents/dive_into_deep_learning.pdf


In [6]:
for filename in os.listdir("ml_documents"):
    filepath = os.path.join("ml_documents", filename)

    print(
        f"{filename} | "
        f"{os.path.getsize(filepath) / (1024 * 1024):.2f} MB"
    )

dive_into_deep_learning.pdf | 42.62 MB
attention_is_all_you_need.pdf | 2.11 MB


In [8]:
documents_text = []

for filename in os.listdir("ml_documents"):
    if filename.endswith(".pdf"):
        filepath = os.path.join("ml_documents", filename)
        pdf = fitz.open(filepath)

        for page_num, page in enumerate(pdf):
            text = page.get_text()

            if text.strip():
                documents_text.append({
                    "text": text,
                    "source": filename,
                    "page": page_num + 1
                })

print("Total pages extracted:", len(documents_text))
print(documents_text[0]["text"][:1000])

Total pages extracted: 1164
Dive into Deep Learning
ASTON ZHANG, ZACHARY C. LIPTON, MU LI, AND ALEXANDER J.
SMOLA



## Create chunks from extracted pages

In [16]:
chunk_size = 150
overlap = 30

chunks = []

for doc in documents_text:
    words = doc["text"].split()

    for start in range(0, len(words), chunk_size - overlap):
        chunk_words = words[start:start + chunk_size]

        if not chunk_words:
            continue

        chunks.append({
            "text": " ".join(chunk_words),
            "source": doc["source"],
            "page": doc["page"]
        })

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 3620
{'text': 'Dive into Deep Learning ASTON ZHANG, ZACHARY C. LIPTON, MU LI, AND ALEXANDER J. SMOLA', 'source': 'dive_into_deep_learning.pdf', 'page': 1}


## Generate Document Embeddings

In [14]:
embedding_model = SentenceTransformer("intfloat/e5-base-v2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully!


In [17]:
chunk_texts = [
    f"passage: {chunk['text']}"
    for chunk in chunks
]

chunk_texts

['passage: Dive into Deep Learning ASTON ZHANG, ZACHARY C. LIPTON, MU LI, AND ALEXANDER J. SMOLA',
 'passage: Contents Preface page xxv Installation xxxiv Notation xxxvii 1 Introduction 1 1.1 A Motivating Example 2 1.2 Key Components 4 1.3 Kinds of Machine Learning Problems 7 1.4 Roots 20 1.5 The Road to Deep Learning 22 1.6 Success Stories 25 1.7 The Essence of Deep Learning 27 1.8 Summary 29 1.9 Exercises 29 2 Preliminaries 30 2.1 Data Manipulation 30 2.1.1 Getting Started 30 2.1.2 Indexing and Slicing 33 2.1.3 Operations 34 2.1.4 Broadcasting 35 2.1.5 Saving Memory 36 2.1.6 Conversion to Other Python Objects 37 2.1.7 Summary 37 2.1.8 Exercises 38 2.2 Data Preprocessing 38 2.2.1 Reading the Dataset 38 2.2.2 Data Preparation 39 2.2.3 Conversion to the Tensor Format 40 2.2.4 Discussion 40 2.2.5 Exercises 40 2.3 Linear Algebra 41 2.3.1 Scalars 41 iii',
 'passage: Linear Algebra 41 2.3.1 Scalars 41 iii',
 'passage: 2.3.2 Vectors 42 2.3.3 Matrices 43 2.3.4 Tensors 44 2.3.5 Basic Propertie

In [23]:
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

Batches:   0%|          | 0/227 [00:00<?, ?it/s]

In [24]:
print("Embeddings shape:", embeddings.shape)

Embeddings shape: (3620, 768)


## Create a FAISS Index

In [25]:
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Store vectors in the index
index.add(embeddings)

In [26]:
print("Total vectors stored:", index.ntotal)

Total vectors stored: 3620


In [27]:
chunks

[{'text': 'Dive into Deep Learning ASTON ZHANG, ZACHARY C. LIPTON, MU LI, AND ALEXANDER J. SMOLA',
  'source': 'dive_into_deep_learning.pdf',
  'page': 1},
 {'text': 'Contents Preface page xxv Installation xxxiv Notation xxxvii 1 Introduction 1 1.1 A Motivating Example 2 1.2 Key Components 4 1.3 Kinds of Machine Learning Problems 7 1.4 Roots 20 1.5 The Road to Deep Learning 22 1.6 Success Stories 25 1.7 The Essence of Deep Learning 27 1.8 Summary 29 1.9 Exercises 29 2 Preliminaries 30 2.1 Data Manipulation 30 2.1.1 Getting Started 30 2.1.2 Indexing and Slicing 33 2.1.3 Operations 34 2.1.4 Broadcasting 35 2.1.5 Saving Memory 36 2.1.6 Conversion to Other Python Objects 37 2.1.7 Summary 37 2.1.8 Exercises 38 2.2 Data Preprocessing 38 2.2.1 Reading the Dataset 38 2.2.2 Data Preparation 39 2.2.3 Conversion to the Tensor Format 40 2.2.4 Discussion 40 2.2.5 Exercises 40 2.3 Linear Algebra 41 2.3.1 Scalars 41 iii',
  'source': 'dive_into_deep_learning.pdf',
  'page': 3},
 {'text': 'Linear Alge

## Create Retriever

In [67]:
def retrieve(query , top_k = 8):
  # Embed query
  query_embedding = embedding_model.encode(
      [f"query: {query}"],
      normalize_embeddings=True,
      convert_to_numpy = True
  ).astype("float32")

  # Search FAISS
  scores, indices = index.search(query_embedding , top_k)

  # Return retrieved chunks
  results = []

  for score , idx in zip(scores[0] , indices[0]):
    if idx == -1:
      continue

    results.append({
        "text" : chunks[idx]['text'],
        "source" : chunks[idx]['source'],
        "page" : chunks[idx]['page'],
        "score" : float(score)
    })

  return results

## LLM

In [83]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


In [84]:
def generate_answer(query, retrieved_chunks):
    if not retrieved_chunks:
        return "I don't know based on the provided documents."

    context = "\n\n".join(
        chunk["text"] for chunk in retrieved_chunks
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI and Machine Learning tutor. "
                "Answer the user's question using the provided context. "
                "If the context does not contain the answer, "
                "say you cannot find the answer in the documents. "
                "Be clear and concise."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {query}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

## Test model

In [85]:
def rag_answer(query, top_k=8):
    retrieved_chunks = retrieve(query, top_k=top_k)

    if not retrieved_chunks:
        return None, "I don't know based on the provided documents."

    answer = generate_answer(query, retrieved_chunks)

    return retrieved_chunks, answer


def run_qa_bot():
    print("Welcome to the RAG Q&A Bot! 🤖")
    print("Ask a question or type 'exit' to quit.\n")

    while True:
        query = input("User: ").strip()

        if query.lower() == "exit":
            print("Goodbye! 👋")
            break

        if not query:
            print("Please enter a valid question.\n")
            continue

        context, answer = rag_answer(query)

        print(f"\nHelpBot: {answer}")

        if context:
            print("\nSources:")
            for chunk in context:
                print(
                    f"- {chunk['source']} | "
                    f"Page: {chunk['page']} | "
                    f"Score: {chunk['score']:.3f}"
                )

        print()


run_qa_bot()

Welcome to the RAG Q&A Bot! 🤖
Ask a question or type 'exit' to quit.

User: Hi

HelpBot: I'm sorry, but I don't see any questions or prompts in your message. Could you please provide more details so I can assist you better?

Sources:
- dive_into_deep_learning.pdf | Page: 129 | Score: 0.841
- dive_into_deep_learning.pdf | Page: 949 | Score: 0.828
- dive_into_deep_learning.pdf | Page: 1086 | Score: 0.823
- dive_into_deep_learning.pdf | Page: 918 | Score: 0.796
- dive_into_deep_learning.pdf | Page: 688 | Score: 0.793
- dive_into_deep_learning.pdf | Page: 942 | Score: 0.788
- dive_into_deep_learning.pdf | Page: 986 | Score: 0.788
- dive_into_deep_learning.pdf | Page: 647 | Score: 0.786

User: What is ML

HelpBot: ML stands for Multi-Layer Perceptron.

Sources:
- dive_into_deep_learning.pdf | Page: 216 | Score: 0.817
- dive_into_deep_learning.pdf | Page: 463 | Score: 0.813
- dive_into_deep_learning.pdf | Page: 149 | Score: 0.810
- dive_into_deep_learning.pdf | Page: 818 | Score: 0.809
- div

## Save models

In [ ]:
# SAVE_DIR = "/content/rag_artifacts"

# os.makedirs(SAVE_DIR, exist_ok=True)

# # 1. Save LLM
# llm.save_pretrained(
#     f"{SAVE_DIR}/llm",
#     safe_serialization=True
# )

# # 2. Save tokenizer
# tokenizer.save_pretrained(
#     f"{SAVE_DIR}/llm"
# )

# # 3. Save embedding model
# embedding_model.save_pretrained(
#     f"{SAVE_DIR}/embedding_model"
# )

# # 4. Save FAISS index
# faiss.write_index(
#     index,
#     f"{SAVE_DIR}/faiss.index"
# )

# # 5. Save chunks and metadata
# with open(f"{SAVE_DIR}/chunks.pkl", "wb") as f:
#     pickle.dump(chunks, f)

# print("All RAG components saved successfully!")